Install Required Libraries


In [1]:
!pip install transformers
!pip install datasets
!pip install peft
!pip install torch
!pip install -qU huggingface_hub
!pip install evaluate
!pip install nltk
!pip install rouge_score
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

Hugging Face Login

In [ ]:
from huggingface_hub import login
login("your_token")

Load Pre-Trained Model & Tokenizer


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "google/gemma-3-1b-pt"

model = AutoModelForCausalLM.from_pretrained(model_name, attn_implementation='eager')
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Load the Dataset

In [15]:
from datasets import load_dataset

dataset = load_dataset("sahil2801/CodeAlpaca-20k", split = "train")

train_size = 0.4
test_size = 0.1 / (1 - train_size)

train_dataset, remaining = dataset.train_test_split(train_size=train_size).values()
test_dataset, _ = remaining.train_test_split(train_size=test_size).values()

print(f"Full Dataset Size: {len(dataset)}, Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json:   0%|          | 0.00/8.06M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

Full Dataset Size: 20022, Train size: 8008, Test size: 2002


A sample

In [16]:
print(train_dataset[0]['output'])
print("**************************")
print(train_dataset[0]['instruction'])
print("**************************")
print(train_dataset[0]['input'])

SELECT name FROM students WHERE last_name = 'Jones';
**************************
Write a SELECT query in MySQL to retrieve the names of all students whose last name is Jones
**************************



Instruction Template Function

In [17]:
def format_prompt(example):
    instruction = example["instruction"]
    input_text = example["input"] if example["input"] else ""
    output_text = example["output"]

    prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Output:\n{output_text}"

    return {"text": prompt}

train_dataset_formatted = train_dataset.map(format_prompt)
test_dataset_formatted = test_dataset.map(format_prompt)

Map:   0%|          | 0/8008 [00:00<?, ? examples/s]

Map:   0%|          | 0/2002 [00:00<?, ? examples/s]

Tokenize the dataset

In [7]:
def preprocess_function(examples):
    inputs = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=128
    )
    inputs["labels"] = inputs["input_ids"].copy()
    return inputs

tokenized__traindataset = train_dataset_formatted.map(preprocess_function, batched=True)

Map:   0%|          | 0/8008 [00:00<?, ? examples/s]

Tokenize the Test dataset

In [18]:
def format_and_tokenize_eval(example):
    instruction = example["instruction"]
    input_text = example["input"] if example["input"] else ""
    output_text = example["output"]

    prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Output:\n"

    input_tokens = tokenizer(prompt, return_tensors="pt", max_length=64, padding="max_length", truncation=True)
    output_tokens = tokenizer(output_text, return_tensors="pt",max_length=64, padding="max_length", truncation=True)

    return {
        "input_ids": input_tokens["input_ids"][0],
        "attention_mask": input_tokens["attention_mask"][0],
        "labels": output_tokens["input_ids"][0],
    }

tokenized_eval_dataset = test_dataset.map(format_and_tokenize_eval, remove_columns=test_dataset.column_names)

Map:   0%|          | 0/2002 [00:00<?, ? examples/s]

Configure Training Hyperparameters

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    num_train_epochs=2,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    fp16=True,
    optim="adamw_8bit",
    logging_steps=100,
    output_dir="outputs",
    logging_dir="logs",
    run_name="sft",
    remove_unused_columns=False,
    weight_decay=0.01,
    push_to_hub=True,
    hub_token="your_token",
    hub_model_id="your_model_id",
    lr_scheduler_type="linear"
)

Initialise the trainer for Full Finetuning

In [ ]:
import torch
import gc

model = model.to("cpu")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized__traindataset,
)
gc.collect()
torch.cuda.empty_cache()
model = torch.compile(model)
model = model.to("cuda")
trainer.train()

trainer.push_to_hub()

PEFT Config

In [8]:
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

peft_model = get_peft_model(model, lora_config)

Initialise the trainer for PEFT Finetuning

In [9]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    optim="adamw_8bit",
    logging_steps=100,
    output_dir="outputs",
    run_name="peft",
    logging_dir="logs",
    remove_unused_columns=False,
    weight_decay=0.01,
    lr_scheduler_type="linear"
)

Initialise the trainer for PEFT Finetuning

In [ ]:
import torch
import gc

model = model.to("cpu")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized__traindataset,
)
gc.collect()
torch.cuda.empty_cache()
peft_model = torch.compile(peft_model)
peft_model = peft_model.to("cuda")
trainer.train()

Pushing Peft Model into Hugging face

In [ ]:
merged_model = peft_model.merge_and_unload()
merged_model.push_to_hub("your_model_id")
tokenizer.push_to_hub("your_model_id")

model.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/devichand/gemma-3-1b-peft-Codealpaca/commit/feae34434a915fe1f19d4566dee06391c8d7993e', commit_message='Upload tokenizer', commit_description='', oid='feae34434a915fe1f19d4566dee06391c8d7993e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/devichand/gemma-3-1b-peft-Codealpaca', endpoint='https://huggingface.co', repo_type='model', repo_id='devichand/gemma-3-1b-peft-Codealpaca'), pr_revision=None, pr_num=None)

Install Merge Kit

In [3]:
!git clone https://github.com/arcee-ai/mergekit.git
!cd mergekit && pip install -qqq -e . --progress-bar off

Cloning into 'mergekit'...
remote: Enumerating objects: 3031, done.
remote: Counting objects: 100% (1218/1218), done.
remote: Compressing objects: 100% (362/362), done.
remote: Total 3031 (delta 1050), reused 856 (delta 856), pack-reused 1813 (from 3)
Receiving objects: 100% (3031/3031), 1.03 MiB | 3.06 MiB/s, done.
Resolving deltas: 100% (2080/2080), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mergekit (pyproject.toml) ... done


## DARE for Full SFT

DARE Configuration

In [ ]:
yaml_config = """
models:
  - model: your_model_id
    parameters:
      density: 0.70
      weight: 1.0

merge_method: dare_linear
base_model: google/gemma-3-1b-pt
parameters:
  int8_mask: true
dtype: bfloat16
"""

with open('config.yaml', 'w', encoding="utf-8") as f:
    f.write(yaml_config)

Merging DARE

In [ ]:
!mergekit-yaml config.yaml merge --copy-tokenizer --cuda --low-cpu-memory

Uploading Model to Hub

In [ ]:
from huggingface_hub import HfApi

MODEL_NAME = "gemma-3-1b-dare-codealpaca"

api = HfApi(token="your_token")

api.create_repo(
    repo_id=f"your_user_id/{MODEL_NAME}",
    repo_type="model",
    exist_ok=True,
)
api.upload_folder(
    repo_id=f"your_user_id/{MODEL_NAME}",
    folder_path="merge",
)

## DARE for PEFT

DARE Configuration

In [ ]:
yaml_config = """
models:
  - model: your_model_id
    parameters:
      density: 0.70
      weight: 1.0

merge_method: dare_linear
base_model: google/gemma-3-1b-pt
parameters:
  int8_mask: true
dtype: bfloat16
"""

with open('config.yaml', 'w', encoding="utf-8") as f:
    f.write(yaml_config)

Merging DARE

In [ ]:
!mergekit-yaml config.yaml merge --copy-tokenizer --cuda --low-cpu-memory

Uploading Model to Hub

In [ ]:
from huggingface_hub import HfApi

MODEL_NAME = "gemma-3-1b-dare-peft-codealpaca"

api = HfApi(token="your_token")

api.create_repo(
    repo_id=f"your_user_id/{MODEL_NAME}",
    repo_type="model",
    exist_ok=True,
)
api.upload_folder(
    repo_id=f"your_user_id/{MODEL_NAME}",
    folder_path="merge",
)

## Evaluation of SFT, PEFT, SFT+DARE and PEFT+DARE

In [9]:
import evaluate
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

Loading Evaluation Metrics

In [10]:
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Loading SFT

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "your_model_id"

model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-pt")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

Loading SFT+DARE

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "your_model_id"    

model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained("your_base_model_id")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

Loading PEFT

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "your_model_id"

model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-pt")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

Loading PEFT+DARE

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "your_model_id"

model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

Evaluation function

In [12]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import DataCollatorWithPadding


def evaluate_model(model, tokenizer, eval_dataset, batch_size=16, device="cuda"):

    collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")
    loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False, collate_fn=collator)
    generated_responses = []
    preferred_responses = []
    qa_prompts = []

    generation_kwargs = {
        "max_new_tokens": 64,
        "top_k": 0.0,
        "top_p": 1.0,
        "do_sample": True,
        "pad_token_id": tokenizer.eos_token_id,
    }

    model.eval()
    model.to(device)

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            generated = model.generate(input_ids=input_ids, attention_mask=attention_mask, **generation_kwargs)
            references = [tokenizer.decode(ids, skip_special_tokens=True) for ids in labels]
            generations = [tokenizer.decode(ids, skip_special_tokens=True) for ids in generated]

            preferred_responses.extend(references)
            generated_responses.extend(generations)

    return generated_responses, preferred_responses

Evaluation of SFT

In [ ]:
generated_responses, preferred_responses = evaluate_model(model, tokenizer, tokenized_eval_dataset)

references = [[nltk.word_tokenize(resp)] for resp in preferred_responses]
candidates = [" ".join(nltk.word_tokenize(resp)) for resp in generated_responses]

results = rouge_metric.compute(predictions=generated_responses, references=preferred_responses)
print("ROUGE Score:",results)
results = bleu_metric.compute(predictions=generated_responses, references=references)
print("BLEU Score:",results)
results = meteor_metric.compute(predictions=generated_responses, references=preferred_responses)
print("METEOR Score:",results)

100%|██████████| 126/126 [09:48<00:00,  4.67s/it]


ROUGE Score: {'rouge1': np.float64(0.1938541784263671), 'rouge2': np.float64(0.07500316189786134), 'rougeL': np.float64(0.16478394938533095), 'rougeLsum': np.float64(0.1862100138255827)}
BLEU Score: {'bleu': 0.0027991561808089287, 'precisions': [0.1377215654889201, 0.0065542822196814595, 0.0009970761415848122, 0.00021170236013986013], 'brevity_penalty': 0.7534108196493129, 'length_ratio': 0.7793353783231084, 'translation_length': 152438, 'reference_length': 195600}
METEOR Score: {'meteor': np.float64(0.20269654734886264)}


Evaluation of SFT+DARE

In [ ]:
generated_responses, preferred_responses = evaluate_model(model, tokenizer, tokenized_eval_dataset)

references = [[nltk.word_tokenize(resp)] for resp in preferred_responses]
candidates = [" ".join(nltk.word_tokenize(resp)) for resp in generated_responses]

results = rouge_metric.compute(predictions=generated_responses, references=preferred_responses)
print("ROUGE Score:",results)
results = bleu_metric.compute(predictions=generated_responses, references=references)
print("BLEU Score:",results)
results = meteor_metric.compute(predictions=generated_responses, references=preferred_responses)
print("METEOR Score:",results)

100%|██████████| 126/126 [09:58<00:00,  4.75s/it]


ROUGE Score: {'rouge1': np.float64(0.25206657059190096), 'rouge2': np.float64(0.10732025470809094), 'rougeL': np.float64(0.22157465131437165), 'rougeLsum': np.float64(0.2413210138798244)}
BLEU Score: {'bleu': 0.0018551655132531928, 'precisions': [0.04871445228355171, 0.002634704244468618, 0.0005846774193548387, 0.0001578427478894897], 'brevity_penalty': 1.0, 'length_ratio': 1.0347852760736196, 'translation_length': 202404, 'reference_length': 195600}
METEOR Score: {'meteor': np.float64(0.1501184955828423)}


Evaluation of PEFT

In [19]:
generated_responses, preferred_responses = evaluate_model(model, tokenizer, tokenized_eval_dataset)

references = [[nltk.word_tokenize(resp)] for resp in preferred_responses]
candidates = [" ".join(nltk.word_tokenize(resp)) for resp in generated_responses]

results = rouge_metric.compute(predictions=generated_responses, references=preferred_responses)
print("ROUGE Score:",results)
results = bleu_metric.compute(predictions=generated_responses, references=references)
print("BLEU Score:",results)
results = meteor_metric.compute(predictions=generated_responses, references=preferred_responses)
print("METEOR Score:",results)

100%|██████████| 126/126 [09:13<00:00,  4.39s/it]


ROUGE Score: {'rouge1': np.float64(0.1539943223446182), 'rouge2': np.float64(0.0585982629645562), 'rougeL': np.float64(0.13128508858522303), 'rougeLsum': np.float64(0.14658926979899853)}
BLEU Score: {'bleu': 0.0022947174962878306, 'precisions': [0.11384277019724826, 0.0032453192510801727, 0.0007456734728283071, 0.00025620643669401724], 'brevity_penalty': 0.791687566267558, 'length_ratio': 0.8106431268475872, 'translation_length': 158227, 'reference_length': 195187}
METEOR Score: {'meteor': np.float64(0.18198402978426972)}


Evaluation of PEFT+DARE

In [23]:
generated_responses, preferred_responses = evaluate_model(model, tokenizer, tokenized_eval_dataset)

references = [[nltk.word_tokenize(resp)] for resp in preferred_responses]
candidates = [" ".join(nltk.word_tokenize(resp)) for resp in generated_responses]

results = rouge_metric.compute(predictions=generated_responses, references=preferred_responses)
print("ROUGE Score:",results)
results = bleu_metric.compute(predictions=generated_responses, references=references)
print("BLEU Score:",results)
results = meteor_metric.compute(predictions=generated_responses, references=preferred_responses)
print("METEOR Score:",results)

100%|██████████| 126/126 [09:29<00:00,  4.52s/it]


ROUGE Score: {'rouge1': np.float64(0.15550387384467837), 'rouge2': np.float64(0.05947333007735721), 'rougeL': np.float64(0.1325778900683971), 'rougeLsum': np.float64(0.1481318834950639)}
BLEU Score: {'bleu': 0.002280568552164021, 'precisions': [0.11476754090441017, 0.0033351361096087976, 0.0007370494540615991, 0.0002511367241196997], 'brevity_penalty': 0.7860646356792577, 'length_ratio': 0.8059860543991147, 'translation_length': 157318, 'reference_length': 195187}
METEOR Score: {'meteor': np.float64(0.18054418866344443)}
